In [1]:
from __future__ import annotations

from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


In [2]:
# -----------------------------------------------------------------------------
# Config
# -----------------------------------------------------------------------------

DATA_URI = "s3://pdga-bronze-266672886271-us-east-2/gold/pdga/wind_effects/model_inputs_hole/event_year=*/tourn_id=*/*.parquet"
AWS_REGION = "us-east-2"

OUTPUT_DIR = Path.cwd() / "notebook_exports"
OUTPUT_DIR.mkdir(exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "round_level_modeling_dataset_v1.parquet"

KEEP_ONLY_WEATHER_READY_ROWS = True

print("DATA_URI:", DATA_URI)
print("OUTPUT_PATH:", OUTPUT_PATH)


DATA_URI: s3://pdga-bronze-266672886271-us-east-2/gold/pdga/wind_effects/model_inputs_hole/event_year=*/tourn_id=*/*.parquet
OUTPUT_PATH: c:\Users\ryanc\dg_wind_effects\wind_impact_analysis\notebook_exports\round_level_modeling_dataset_v1.parquet


In [3]:
# -----------------------------------------------------------------------------
# DuckDB setup
# -----------------------------------------------------------------------------

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute(f"SET s3_region='{AWS_REGION}'")

print("DuckDB ready.")


DuckDB ready.


In [4]:
# -----------------------------------------------------------------------------
# Quick sanity check: count source rows
# -----------------------------------------------------------------------------

row_count_sql = f"""
SELECT COUNT(*) AS hole_rows
FROM read_parquet('{DATA_URI}', hive_partitioning=true)
"""

source_row_count = con.execute(row_count_sql).df()
source_row_count


,hole_rows
0,12093744


In [5]:
# -----------------------------------------------------------------------------
# Aggregate hole-level source to round level
# -----------------------------------------------------------------------------

weather_filter_sql = "AND weather_available_flag = TRUE" if KEEP_ONLY_WEATHER_READY_ROWS else ""

round_sql = f"""
WITH hole_rows AS (
    SELECT
        CAST(event_year AS BIGINT) AS event_year,
        CAST(tourn_id AS BIGINT) AS tourn_id,
        CAST(round_number AS BIGINT) AS round_number,
        CAST(hole_number AS BIGINT) AS hole_number,
        CAST(player_key AS VARCHAR) AS player_key,
        CAST(player_rating AS DOUBLE) AS player_rating,
        CAST(course_id AS VARCHAR) AS course_id,
        CAST(layout_id AS VARCHAR) AS layout_id,
        CAST(hole_length AS DOUBLE) AS hole_length,
        CAST(hole_par AS DOUBLE) AS hole_par,
        CAST(actual_strokes AS DOUBLE) AS actual_strokes,
        CAST(division AS VARCHAR) AS division,
        CAST(weather_available_flag AS BOOLEAN) AS weather_available_flag,

        COALESCE(CAST(wx_wind_speed_mps AS DOUBLE), CAST(feature_wind_speed_mps AS DOUBLE)) AS wind_speed_mps,
        COALESCE(CAST(wx_wind_gust_mps AS DOUBLE), CAST(feature_wind_gust_mps AS DOUBLE)) AS wind_gust_mps,
        COALESCE(CAST(wx_wind_dir_deg AS DOUBLE), CAST(feature_wind_dir_deg AS DOUBLE)) AS wind_dir_deg,
        COALESCE(CAST(wx_temperature_c AS DOUBLE), CAST(feature_temp_c AS DOUBLE)) AS temp_c,
        COALESCE(CAST(wx_precip_mm AS DOUBLE), CAST(feature_precip_mm AS DOUBLE)) AS precip_mm,
        COALESCE(CAST(wx_pressure_hpa AS DOUBLE), CAST(feature_pressure_hpa AS DOUBLE)) AS pressure_hpa,
        COALESCE(CAST(wx_relative_humidity_pct AS DOUBLE), CAST(feature_humidity_pct AS DOUBLE)) AS humidity_pct
    FROM read_parquet('{DATA_URI}', hive_partitioning=true)
),
filtered AS (
    SELECT *
    FROM hole_rows
    WHERE
        event_year IS NOT NULL
        AND tourn_id IS NOT NULL
        AND round_number IS NOT NULL
        AND hole_number IS NOT NULL
        AND player_key IS NOT NULL
        AND actual_strokes BETWEEN 1 AND 15
        AND player_rating BETWEEN 500 AND 1100
        AND hole_length BETWEEN 30 AND 2000
        AND hole_par BETWEEN 2 AND 6
        AND wind_speed_mps IS NOT NULL
        AND wind_gust_mps IS NOT NULL
        AND wind_dir_deg IS NOT NULL
        AND temp_c IS NOT NULL
        AND precip_mm IS NOT NULL
        {weather_filter_sql}
),
round_level AS (
    SELECT
        event_year,
        tourn_id,
        round_number,
        player_key,

        ANY_VALUE(course_id) AS course_id,
        ANY_VALUE(layout_id) AS layout_id,
        ANY_VALUE(division) AS division,
        ANY_VALUE(player_rating) AS player_rating,

        SUM(actual_strokes) AS actual_round_strokes,
        COUNT(DISTINCT hole_number) AS hole_count,

        SUM(hole_length) AS round_total_hole_length,
        AVG(hole_length) AS round_avg_hole_length,
        SUM(hole_par) AS round_total_par,
        AVG(hole_par) AS round_avg_hole_par,

        AVG(wind_speed_mps) AS round_wind_speed_mps_mean,
        MAX(wind_speed_mps) AS round_wind_speed_mps_max,
        AVG(wind_gust_mps) AS round_wind_gust_mps_mean,
        MAX(wind_gust_mps) AS round_wind_gust_mps_max,

        AVG(temp_c) AS round_temp_c_mean,
        SUM(precip_mm) AS round_precip_mm_sum,
        AVG(precip_mm) AS round_precip_mm_mean,
        AVG(pressure_hpa) AS round_pressure_hpa_mean,
        AVG(humidity_pct) AS round_humidity_pct_mean
    FROM filtered
    GROUP BY 1,2,3,4
)
SELECT
    *,
    actual_round_strokes - round_total_par AS round_strokes_over_par,
    round_total_hole_length / NULLIF(round_total_par, 0) AS round_length_over_par,
    round_wind_speed_mps_mean * round_total_hole_length AS wind_x_round_length,
    round_wind_gust_mps_mean * round_total_hole_length AS gust_x_round_length,
    round_wind_speed_mps_mean * player_rating AS wind_x_player_rating
FROM round_level
"""

round_level_df = con.execute(round_sql).df()

print("Round-level dataframe shape:", round_level_df.shape)
round_level_df.head()


Round-level dataframe shape: (419842, 28)


,event_year,tourn_id,round_number,player_key,course_id,layout_id,division,player_rating,actual_round_strokes,hole_count,round_total_hole_length,round_avg_hole_length,round_total_par,round_avg_hole_par,round_wind_speed_mps_mean,round_wind_speed_mps_max,round_wind_gust_mps_mean,round_wind_gust_mps_max,round_temp_c_mean,round_precip_mm_sum,round_precip_mm_mean,round_pressure_hpa_mean,round_humidity_pct_mean,round_strokes_over_par,round_length_over_par,wind_x_round_length,gust_x_round_length,wind_x_player_rating
0,2025,90004,1,PDGA#240866,248066,731340,MA4,777.0,60.0,18,4262.0,236.777778,54.0,3.000000,2.99,2.99,5.3,5.3,15.6,0.0,0.0,1018.5,82.0,6.0,78.925926,12743.38,22588.6,2323.23
1,2025,90022,1,PDGA#139372,303386,708703,MPO,944.0,65.0,22,6640.0,301.818182,69.0,3.136364,0.69,0.69,3.0,3.0,3.7,0.0,0.0,1009.0,83.0,-4.0,96.231884,4581.60,19920.0,651.36
2,2025,90022,1,PDGA#178707,303386,708703,FPO,921.0,67.0,22,6640.0,301.818182,69.0,3.136364,1.38,1.38,2.7,2.7,1.1,0.0,0.0,1009.0,91.0,-2.0,96.231884,9163.20,17928.0,1270.98
3,2025,90022,1,PDGA#235527,303386,708703,MA2,861.0,72.0,22,6640.0,301.818182,69.0,3.136364,1.25,1.25,2.3,2.3,2.2,0.0,0.0,1009.7,89.0,3.0,96.231884,8300.00,15272.0,1076.25
4,2025,90022,2,PDGA#208891,303386,708709,MA50,887.0,79.0,22,7337.0,333.500000,70.0,3.181818,1.71,1.71,4.2,4.2,5.5,0.0,0.0,1004.7,77.0,9.0,104.814286,12546.27,30815.4,1516.77


In [6]:
# -----------------------------------------------------------------------------
# Basic validation
# -----------------------------------------------------------------------------

validation_summary = pd.DataFrame(
    [
        {"metric": "rows", "value": len(round_level_df)},
        {"metric": "unique_tournaments", "value": round_level_df["tourn_id"].nunique()},
        {"metric": "unique_courses", "value": round_level_df["course_id"].nunique(dropna=True)},
        {"metric": "unique_layouts", "value": round_level_df["layout_id"].nunique(dropna=True)},
        {"metric": "unique_players", "value": round_level_df["player_key"].nunique()},
        {"metric": "missing_division_rows", "value": round_level_df["division"].isna().sum()},
        {"metric": "missing_layout_rows", "value": round_level_df["layout_id"].isna().sum()},
    ]
)

validation_summary


,metric,value
0,rows,419842
1,unique_tournaments,6115
2,unique_courses,3020
3,unique_layouts,13033
4,unique_players,44129
5,missing_division_rows,0
6,missing_layout_rows,0


In [7]:
# -----------------------------------------------------------------------------
# Inspect distributions
# -----------------------------------------------------------------------------

round_level_df[
    [
        "actual_round_strokes",
        "round_strokes_over_par",
        "player_rating",
        "round_total_hole_length",
        "round_total_par",
        "round_wind_speed_mps_mean",
        "round_wind_gust_mps_mean",
        "round_temp_c_mean",
        "round_precip_mm_mean",
    ]
].describe()


,actual_round_strokes,round_strokes_over_par,player_rating,round_total_hole_length,round_total_par,round_wind_speed_mps_mean,round_wind_gust_mps_mean,round_temp_c_mean,round_precip_mm_mean
count,419842.000000,419842.000000,419842.000000,419842.000000,419842.000000,419842.000000,419842.000000,419842.000000,419842.000000
mean,62.421394,2.549266,888.736091,5345.342681,59.872128,2.715734,5.941519,15.357337,0.088096
std,10.451388,7.568406,65.141904,2017.244412,6.619257,1.658357,3.247156,8.207632,0.516517
min,2.000000,-43.000000,501.000000,75.000000,3.000000,0.000000,0.200000,-26.100000,0.000000
25%,56.000000,-3.000000,856.000000,4506.000000,56.000000,1.490000,3.600000,10.100000,0.000000
50%,61.000000,2.000000,897.000000,5601.000000,59.000000,2.440000,5.400000,15.900000,0.000000
75%,68.000000,7.000000,933.000000,6542.000000,62.000000,3.640000,7.700000,21.300000,0.000000
max,227.000000,145.000000,1059.000000,14114.000000,127.000000,12.700000,25.000000,45.300000,18.600000


In [8]:
# -----------------------------------------------------------------------------
# Save locally for downstream modeling notebooks
# -----------------------------------------------------------------------------

round_level_df.to_parquet(OUTPUT_PATH, index=False)

print("Saved round-level dataset to:", OUTPUT_PATH)


Saved round-level dataset to: c:\Users\ryanc\dg_wind_effects\wind_impact_analysis\notebook_exports\round_level_modeling_dataset_v1.parquet
